In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

from src.data import make_splits
from src.esm_embed import embed_sequences
from src.models import EmbeddingClassifier
from src.train import set_seed, train_cnn, predict

device = "cuda"
train, val, test = make_splits("../data/deeploc_data.fasta")
print(len(train), len(val), len(test))          # should print 8867 2218 2773

le = LabelEncoder().fit(train.location)
y_train, y_val = le.transform(train.location), le.transform(val.location)
class_names = list(le.classes_)

C:\dev\protein-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8867 2218 2773


In [2]:
MODEL = "facebook/esm2_t12_35M_UR50D"
cache = Path("../outputs")

def get_embeddings(df, name):
    path = cache / f"esm2_35M_{name}.npy"
    if path.exists():
        return np.load(path)
    emb = embed_sequences(df.seq.tolist(), MODEL, device, max_len=1000, batch_size=16)
    np.save(path, emb)
    return emb

E_train = get_embeddings(train, "train")
E_val = get_embeddings(val, "val")
print(E_train.shape, E_val.shape)               # (8867, 480) (2218, 480)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2897.88it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1776.10it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.laye

(8867, 480) (2218, 480)


In [3]:
rows = []
for seed in [0, 1, 2]:
    print(f"===== seed {seed} =====")
    set_seed(seed)
    head = EmbeddingClassifier(in_dim=E_train.shape[1], n_classes=len(class_names))
    head, _ = train_cnn(head, E_train, y_train, E_val, y_val, device, seed,
                        epochs=50, patience=5, lr=1e-3, batch_size=64)
    pred = predict(head, E_val, device)
    rows.append({"model": "esm2_35M_frozen+mlp", "seed": seed,
                 "val_acc": accuracy_score(y_val, pred),
                 "val_macro_f1": f1_score(y_val, pred, average="macro")})
    print(rows[-1])

res = pd.DataFrame(rows)
display(res.groupby("model")[["val_acc", "val_macro_f1"]].agg(["mean", "std"]).round(3))
res.to_csv("../outputs/frozen_esm2_results.csv", index=False)

===== seed 0 =====
epoch  1  loss 1.1797  val acc 0.699  val macro-F1 0.489
epoch  2  loss 0.8784  val acc 0.711  val macro-F1 0.527
epoch  3  loss 0.7889  val acc 0.728  val macro-F1 0.569
epoch  4  loss 0.7460  val acc 0.736  val macro-F1 0.561
epoch  5  loss 0.7030  val acc 0.741  val macro-F1 0.606
epoch  6  loss 0.6728  val acc 0.748  val macro-F1 0.589
epoch  7  loss 0.6412  val acc 0.747  val macro-F1 0.613
epoch  8  loss 0.6134  val acc 0.747  val macro-F1 0.625
epoch  9  loss 0.5845  val acc 0.741  val macro-F1 0.625
epoch 10  loss 0.5638  val acc 0.750  val macro-F1 0.636
epoch 11  loss 0.5469  val acc 0.745  val macro-F1 0.632
epoch 12  loss 0.5217  val acc 0.749  val macro-F1 0.649
epoch 13  loss 0.5025  val acc 0.745  val macro-F1 0.642
epoch 14  loss 0.4833  val acc 0.739  val macro-F1 0.645
epoch 15  loss 0.4569  val acc 0.746  val macro-F1 0.645
epoch 16  loss 0.4467  val acc 0.746  val macro-F1 0.639
epoch 17  loss 0.4320  val acc 0.742  val macro-F1 0.643
early stoppi

val_acc        val_macro_f1       
                       mean    std         mean    std
model                                                 
esm2_35M_frozen+mlp    0.75  0.004        0.655  0.008

In [4]:
from sklearn.metrics import classification_report
print(classification_report(y_val, pred, target_names=class_names, digits=3, zero_division=0))

                       precision    recall  f1-score   support

        Cell.membrane      0.798     0.776     0.787       214
            Cytoplasm      0.622     0.635     0.629       406
Endoplasmic.reticulum      0.750     0.630     0.685       138
        Extracellular      0.878     0.908     0.893       316
      Golgi.apparatus      0.733     0.386     0.506        57
     Lysosome/Vacuole      0.533     0.308     0.390        52
        Mitochondrion      0.770     0.719     0.744       242
              Nucleus      0.761     0.872     0.813       647
           Peroxisome      0.778     0.280     0.412        25
              Plastid      0.802     0.769     0.785       121

             accuracy                          0.755      2218
            macro avg      0.742     0.628     0.664      2218
         weighted avg      0.752     0.755     0.749      2218

